In [1]:
# Sys path
from sys import path
from pathlib import Path

module_path = str(Path.cwd().parents[1])

if module_path not in path:
    path.append(module_path)
    
path.append(module_path + '\\functions')


# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', 1000)
pd.set_option('display.max_rows', 1000)

import data_preparation
from classification import LR, KNN, RF, XGB

from sklearn.metrics import f1_score, roc_auc_score

import time
import pickle


# Dataset

In [3]:
df, X, y = data_preparation.load_dataset(module_path + '\\dataset\\multi_attack_FDI_SLC.csv')

# Data Analysis

In [5]:
num = y.value_counts()
num =list(np.array(num))
names = ['FD', 'SLC']

# ML models

### 1. All features

In [60]:
X_train, X_test, y_train, y_test = data_preparation.split_data(X, y, test_size=0.2, random_state=123)

train = 1

if train==1:
    # training
    train_LR = -time.time()
    LR(X_train, y_train, normalize=False, save_model='\\SE_identification\\classification_models\\lr_model.pickle', save_param='\\SE_identification\\classification_models\\lr_parametres.pickle')
    train_LR += time.time()
    print('LR')

    train_KNN = -time.time()
    KNN(X_train, y_train, normalize=False, save_model='\\SE_identification\\classification_models\\knn_model.pickle', save_param='\\SE_identification\\classification_models\\knn_parametres.pickle')
    train_KNN += time.time()
    print('KNN')

    train_RF = -time.time()
    RF(X_train, y_train, normalize=False, save_model='\\SE_identification\\classification_models\\rf_model.pickle', save_param='\\SE_identification\\classification_models\\rf_parametres.pickle')
    train_RF += time.time()
    print('RF')
    
    train_XGB = -time.time()
    XGB(X_train, y_train, normalize=False, save_model='\\SE_identification\\classification_models\\xgb_model.pickle', save_param='\\SE_identification\\classification_models\\xgb_parametres.pickle', 
        save_encoder='\\SE_identification\\classification_models\\xgb_label_encoder.pickle')
    train_XGB += time.time()
    print('XGB')
    
    train_time = data_preparation.to_dict(train_LR, train_KNN, train_RF, train_XGB)
    data_preparation.save_model(train_time, 'time/train_time')
    
train_time = data_preparation.load_model('time/train_time')
M1_score = data_preparation.load_model('score/macro_score')
    
# load models
lr = pickle.load(open(module_path + '\\multi_attack\\SE_identification\\classification_models\\lr_model.pickle', 'rb'))
knn = pickle.load(open(module_path + '\\multi_attack\\SE_identification\\classification_models\\knn_model.pickle', 'rb'))
rf = pickle.load(open(module_path + '\\multi_attack\\SE_identification\\classification_models\\rf_model.pickle', 'rb'))
xgb = pickle.load(open(module_path + '\\multi_attack\\SE_identification\\classification_models\\xgb_model.pickle', 'rb'))

# 加载 encoder
le = pickle.load(open(module_path + '\\multi_attack\\SE_identification\\classification_models\\xgb_label_encoder.pickle', 'rb'))

# load parameters
lr_param = pickle.load(open(module_path + '\\multi_attack\\SE_identification\\classification_models\\lr_parametres.pickle', 'rb'))
knn_param = pickle.load(open(module_path + '\\multi_attack\\SE_identification\\classification_models\\knn_parametres.pickle', 'rb'))
rf_param = pickle.load(open(module_path + '\\multi_attack\\SE_identification\\classification_models\\rf_parametres.pickle', 'rb'))
xgb_param = pickle.load(open(module_path + '\\multi_attack\\SE_identification\\classification_models\\xgb_parametres.pickle', 'rb'))
    
# prediction
test_LR = -time.time()
y_pred_lr = lr.predict(X_test)
test_LR += time.time()
    
test_KNN = -time.time()
y_pred_knn = knn.predict(X_test)
test_KNN += time.time()
    
test_RF = -time.time()
y_pred_rf = rf.predict(X_test)
test_RF += time.time()
    
test_XGB = -time.time()
y_pred_xgb = xgb.predict(X_test)
test_XGB += time.time()

# 转换为原始标签
y_pred_xgb = le.inverse_transform(y_pred_xgb)


LR
KNN
RF
XGB


In [62]:
print('##############')
print('F1 score:')
print('LR', M1_score['LR']) 
print('KNN', M1_score['KNN']) 
print('RF', M1_score['RF']) 
print('XGB', M1_score['XGB']) 

print('##############')
print('Training Time:')
print('LR', train_time['LR'], 'sec')
print('KNN', train_time['KNN'], 'sec')
print('RF', train_time['RF'], 'sec')
print('XGB', train_time['XGB'], 'sec')

print('##############')
print('Testing Time:')
print('LR', test_LR, 'sec')
print('KNN', test_KNN, 'sec')
print('RF', test_RF, 'sec')
print('XGB', test_XGB, 'sec')

print('##############')
print('Parameters:')
print('LR:', lr_param)
print('KNN:', knn_param)
print('RF:', rf_param)
print('XGB:', xgb_param)

##############
F1 score:
LR 82.37462916384234
KNN 95.63827419253904
RF 96.38174529069246
XGB 97.64927381540272
##############
Training Time:
LR 149.13305234909058 sec
KNN 8.257017850875854 sec
RF 163.48820543289185 sec
XGB 87.13357758522034 sec
##############
Testing Time:
LR 0.005001068115234375 sec
KNN 0.047532081604003906 sec
RF 0.16903972625732422 sec
XGB 0.02400374412536621 sec
##############
Parameters:
LR: ['newton-cg', 'none', 8.326365835633833]
KNN: [3, 'distance']
RF: [734, 8, 3, 6]
XGB: [725, 14, 0.1624630235090775, 0.5809759524363649, 0.5491294434181543]


-----------------------

## PCA

In [56]:
import os
import time
import pickle
import numpy as np

from sklearn.decomposition import PCA
from sklearn.metrics import f1_score

train = 1

model_dir = os.path.join(
    module_path,
    'multi_attack',
    'SE_identification',
    'classification_models'
)

os.makedirs(
    model_dir,
    exist_ok=True
)

pca_path = os.path.join(
    model_dir,
    'pca_model.pickle'
)


X_train_raw, X_test_raw, y_train, y_test = (
    data_preparation.split_data(
        X,
        y,
        test_size=0.2,
        random_state=123
    )
)

y_train = np.asarray(y_train).ravel()
y_test = np.asarray(y_test).ravel()


if train == 1:

    # Multi-node论文设置：保留约95%的累计方差
    pca = PCA(
        n_components=0.95
    )

    X_train = pca.fit_transform(
        X_train_raw
    )

    X_test = pca.transform(
        X_test_raw
    )

    with open(pca_path, 'wb') as f:
        pickle.dump(pca, f)

else:

    with open(pca_path, 'rb') as f:
        pca = pickle.load(f)

    X_train = pca.transform(
        X_train_raw
    )

    X_test = pca.transform(
        X_test_raw
    )




if train == 1:

    # --------------------------------------------------------
    # LR
    # --------------------------------------------------------

    train_LR = -time.time()

    LR(
        X_train,
        y_train,
        normalize=False,
        save_model=(
            '\\SE_identification'
            '\\classification_models'
            '\\lr_model_pca.pickle'
        ),
        save_param=(
            '\\SE_identification'
            '\\classification_models'
            '\\lr_parametres_pca.pickle'
        )
    )

    train_LR += time.time()
    print('LR')


    # --------------------------------------------------------
    # KNN
    # --------------------------------------------------------

    train_KNN = -time.time()

    KNN(
        X_train,
        y_train,
        normalize=False,
        save_model=(
            '\\SE_identification'
            '\\classification_models'
            '\\knn_model_pca.pickle'
        ),
        save_param=(
            '\\SE_identification'
            '\\classification_models'
            '\\knn_parametres_pca.pickle'
        )
    )

    train_KNN += time.time()
    print('KNN')


    # --------------------------------------------------------
    # RF
    # --------------------------------------------------------

    train_RF = -time.time()

    RF(
        X_train,
        y_train,
        normalize=False,
        save_model=(
            '\\SE_identification'
            '\\classification_models'
            '\\rf_model_pca.pickle'
        ),
        save_param=(
            '\\SE_identification'
            '\\classification_models'
            '\\rf_parametres_pca.pickle'
        )
    )

    train_RF += time.time()
    print('RF')


    # --------------------------------------------------------
    # XGB
    # --------------------------------------------------------

    train_XGB = -time.time()

    XGB(
        X_train,
        y_train,
        normalize=False,
        save_model=(
            '\\SE_identification'
            '\\classification_models'
            '\\xgb_model_pca.pickle'
        ),
        save_param=(
            '\\SE_identification'
            '\\classification_models'
            '\\xgb_parametres_pca.pickle'
        ),
        save_encoder=(
            '\\SE_identification'
            '\\classification_models'
            '\\xgb_label_encoder_pca.pickle'
        )
    )

    train_XGB += time.time()
    print('XGB')


    train_time = data_preparation.to_dict(
        train_LR,
        train_KNN,
        train_RF,
        train_XGB
    )

    data_preparation.save_model(
        train_time,
        'time/train_time_pca'
    )


train_time = data_preparation.load_model(
    'time/train_time_pca'
)

with open(
    os.path.join(
        model_dir,
        'lr_model_pca.pickle'
    ),
    'rb'
) as f:
    lr = pickle.load(f)


with open(
    os.path.join(
        model_dir,
        'knn_model_pca.pickle'
    ),
    'rb'
) as f:
    knn = pickle.load(f)


with open(
    os.path.join(
        model_dir,
        'rf_model_pca.pickle'
    ),
    'rb'
) as f:
    rf = pickle.load(f)


with open(
    os.path.join(
        model_dir,
        'xgb_model_pca.pickle'
    ),
    'rb'
) as f:
    xgb = pickle.load(f)


with open(
    os.path.join(
        model_dir,
        'xgb_label_encoder_pca.pickle'
    ),
    'rb'
) as f:
    le = pickle.load(f)

with open(
    os.path.join(
        model_dir,
        'lr_parametres_pca.pickle'
    ),
    'rb'
) as f:
    lr_param = pickle.load(f)


with open(
    os.path.join(
        model_dir,
        'knn_parametres_pca.pickle'
    ),
    'rb'
) as f:
    knn_param = pickle.load(f)


with open(
    os.path.join(
        model_dir,
        'rf_parametres_pca.pickle'
    ),
    'rb'
) as f:
    rf_param = pickle.load(f)


with open(
    os.path.join(
        model_dir,
        'xgb_parametres_pca.pickle'
    ),
    'rb'
) as f:
    xgb_param = pickle.load(f)


# LR
test_LR = -time.time()

y_pred_lr = lr.predict(
    X_test
)

test_LR += time.time()


# KNN
test_KNN = -time.time()

y_pred_knn = knn.predict(
    X_test
)

test_KNN += time.time()


# RF
test_RF = -time.time()

y_pred_rf = rf.predict(
    X_test
)

test_RF += time.time()


# XGB
test_XGB = -time.time()

y_pred_xgb = xgb.predict(
    X_test
)

test_XGB += time.time()


y_pred_xgb = le.inverse_transform(
    np.asarray(y_pred_xgb)
    .ravel()
    .astype(int)
)

M1_score = data_preparation.load_model('score/macro_score_pca')

macro_f1_lr = f1_score(
    y_test,
    y_pred_lr,
    average='macro'
)

macro_f1_knn = f1_score(
    y_test,
    y_pred_knn,
    average='macro'
)

macro_f1_rf = f1_score(
    y_test,
    y_pred_rf,
    average='macro'
)

macro_f1_xgb = f1_score(
    y_test,
    y_pred_xgb,
    average='macro'
)

print('##############')
print('F1 score:')

print('LR', M1_score['LR']) 
print('KNN', M1_score['KNN']) 
print('RF', M1_score['RF']) 
print('XGB', M1_score['XGB']) 


print('##############')
print('Training Time:')

print('LR', train_time['LR'], 'sec')
print('KNN', train_time['KNN'], 'sec')
print('RF', train_time['RF'], 'sec')
print('XGB', train_time['XGB'], 'sec')


print('##############')
print('Testing Time:')

print('LR', test_LR, 'sec')
print('KNN', test_KNN, 'sec')
print('RF', test_RF, 'sec')
print('XGB', test_XGB, 'sec')


print('##############')
print('Parameters:')

print('LR:', lr_param)
print('KNN:', knn_param)
print('RF:', rf_param)
print('XGB:', xgb_param)

LR
KNN
RF
XGB
##############
F1 score:
LR 72.39186527481367
KNN 93.74261859370425
RF 95.79638152490713
XGB 96.60372829173829
##############
Training Time:
LR 6.566621541976929 sec
KNN 2.9932193756103516 sec
RF 46.49760866165161 sec
XGB 17.427887678146362 sec
##############
Testing Time:
LR 0.0010001659393310547 sec
KNN 0.007001161575317383 sec
RF 0.09856510162353516 sec
XGB 0.001995086669921875 sec
##############
Parameters:
LR: ['newton-cg', 'none', 8.326365835633833]
KNN: [3, 'distance']
RF: [446, 13, 4, 9]
XGB: [899, 8, 0.322282178478741, 0.6317370805153809, 0.7835905325881662]
